In [2]:
import os

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\Projects\\Kidney-Disease-Classification-Deep-Learning-Project'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    source_dir: Path

In [6]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)



    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            source_dir=config.source_dir
        )

        return data_transformation_config

In [8]:
import os
import shutil
from cnnClassifier import logger
from sklearn.model_selection import train_test_split

In [9]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config



    def transform_data(self):
        classes = os.listdir(self.config.source_dir)

        for class_name in classes:
            class_path = os.path.join(self.config.source_dir, class_name)
            if not os.path.isdir(class_path):
                continue
            images = os.listdir(class_path)
            train_images, temp_images = train_test_split(images, test_size=0.3, random_state=42)
            val_images, test_images = train_test_split(temp_images, test_size=0.5, random_state=42)
            for split, split_images in [("train", train_images),("val", val_images),("test", test_images)]:
                destination = os.path.join(self.config.root_dir, split, class_name)
                os.makedirs(destination, exist_ok=True)
                for image in split_images:
                    shutil.copy2(
                        os.path.join(class_path, image),
                        os.path.join(destination, image)
                    )


In [10]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.transform_data()
except Exception as e:
    raise e

[2026-09-06 16:09:57,329: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-09-06 16:09:57,335: INFO: common: yaml file: params.yaml loaded successfully]
[2026-09-06 16:09:57,338: INFO: common: created directory at: artifacts/data_transformation]
